# Phase 1: Simple EDA & Vigilance Audit (BothBosu Dataset)
We are using `BothBosu/multi-agent-scam-conversation`. This notebook performs a simple EDA (label balance, sequence lengths) and our strict vigilance checks (overlap, manual register inspection).


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sys
sys.path.append("..")
from src.eda_utils import get_overlap_count, get_quote_artifact_stats

try:
    df_train = pd.read_csv("../data/raw/v2_composite_train.csv")
    df_test = pd.read_csv("../data/raw/v2_composite_test.csv")
    print(f"Loaded {len(df_train)} train rows, {len(df_test)} test rows.")
except Exception as e:
    print("Error loading data! Run scripts/download_v2.py first.")


## 1. Simple EDA: Class Distribution


In [ ]:
if "df_train" in locals():
    dist = df_train['label'].value_counts(normalize=True) * 100
    print("Training Set Label Distribution:\n", dist)
    
    plt.figure(figsize=(6, 4))
    sns.countplot(data=df_train, x="label")
    plt.title("Class Balance")
    plt.show()


## 2. Train/Test Leakage Check

In [ ]:
if "df_train" in locals():
    overlap = get_overlap_count(df_train, df_test, text_col="text")
    print(f"Overlapping rows between train and test: {overlap}")


## 3. Structural Artifact Audit

In [ ]:
if "df_train" in locals():
    stats = get_quote_artifact_stats(df_train, text_col="text", label_col="label")
    display(stats)


## 4. Length Distribution (Proving Long Context)

In [ ]:
if "df_train" in locals():
    df_train["word_count"] = df_train["text"].apply(lambda x: len(str(x).split()))
    
    plt.figure(figsize=(10, 5))
    sns.histplot(data=df_train, x="word_count", hue="label", bins=50, kde=True)
    plt.axvline(x=512, color="red", linestyle="--", label="DistilBERT Limit (512)")
    plt.title("Transcript Length Distribution")
    plt.legend()
    plt.show()
    
    pct_long = (df_train["word_count"] > 512).mean() * 100
    print(f"Percentage of training set exceeding 512 words: {pct_long:.2f}%")


## 5. Source-Register Audit (Manual Inspection)
Manually verify they share the same conversational register.

In [ ]:
if "df_train" in locals():
    print("--- LEGITIMATE (LABEL 0) ---")
    legit_sample = df_train[df_train["label"] == 0].sample(min(3, len(df_train[df_train["label"] == 0])))
    for idx, row in legit_sample.iterrows():
        print(f"{str(row['text'])[:500]}...\n")
        
    print("\n--- SCAM (LABEL 1) ---")
    scam_sample = df_train[df_train["label"] == 1].sample(min(3, len(df_train[df_train["label"] == 1])))
    for idx, row in scam_sample.iterrows():
        print(f"{str(row['text'])[:500]}...\n")
